In [1]:
#   CENÁRIO A — COMPLETO
#   Técnicas: Linear, Random Forest, Decision Tree, AdaBoost, Gradient Boosting
# ===============================

import sqlite3
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
# 1) Ler base de dados
# ===============================

db_path = r"C:\Users\beasa\Desktop\AASE\ScreenTimevsMentalWellness.db"
table_main = "ScreenTimevsMentalWellness"

conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {table_main}", conn)
conn.close()

target = "mental_wellness_index_0_100"

if target not in df.columns:
    raise Exception(f"Coluna alvo {target} não existe na BD.")

y = df[target]
mask = y.notna()
df = df[mask]
y = y[mask]

In [3]:
# 2) Features do Cenário A
# ===============================

features_A = [
    "screen_time_hours",
    "work_screen_hours",
    "leisure_screen_hours",
    "sleep_hours",
    "sleep_quality_1_5",
    "exercise_minutes_per_week",
    "social_hours_per_week",
    "stress_level_0_10",
    "productivity_0_100",
    "occupation_Desempregado",
    "occupation_Empregado",
    "occupation_Estudante",
    "occupation_Reformado",
    "work_mode_Hybrid",
    "work_mode_In-person",
    "work_mode_Remote",
    "age_Jovem_16-25",
    "age_Adulto_25-50",
    "age_Senior_50-60"
]

features_A = [c for c in features_A if c in df.columns]
X = df[features_A].fillna(df[features_A].mean())

print("Features usadas no Cenário A:", features_A)


Features usadas no Cenário A: ['screen_time_hours', 'work_screen_hours', 'leisure_screen_hours', 'sleep_hours', 'sleep_quality_1_5', 'exercise_minutes_per_week', 'social_hours_per_week', 'stress_level_0_10', 'productivity_0_100', 'occupation_Desempregado', 'occupation_Empregado', 'occupation_Estudante', 'occupation_Reformado', 'work_mode_Hybrid', 'work_mode_In-person', 'work_mode_Remote', 'age_Jovem_16-25', 'age_Adulto_25-50', 'age_Senior_50-60']


In [4]:
# 3) Modelos — agora com 5 algoritmos
# ===============================

scaler = StandardScaler()

models = {
    "LinearRegression": Pipeline([
        ("scaler", scaler),
        ("model", LinearRegression())
    ]),

    "RandomForest": Pipeline([
        ("scaler", scaler),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1))
    ]),

    "DecisionTree": Pipeline([
        ("scaler", scaler),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("scaler", scaler),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4),
            n_estimators=200,
            random_state=42))
    ]),

    "GradientBoosting": Pipeline([
        ("scaler", scaler),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42))
    ])
}


In [5]:
# 4) Cross-Validation 10-fold
# ===============================

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print("\n===== CENÁRIO A — 5 Técnicas de Regressão =====\n")

results = []

for name, pipe in models.items():
    print(f"A avaliar modelo: {name}...")
    
    cv = cross_validate(
        pipe, X, y, cv=kfold,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }
    )

    mae = -cv["test_MAE"].mean()
    rmse = -cv["test_RMSE"].mean()
    r2 = cv["test_R2"].mean()

    print(f"Modelo: {name}")
    print(f" MAE:  {mae:.3f}")
    print(f" RMSE: {rmse:.3f}")
    print(f" R2:   {r2:.3f}\n")

    results.append({
        "Cenário": "A",
        "Modelo": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

df_results = pd.DataFrame(results)
print("\nResultados finais Cenário A:")
print(df_results)


===== CENÁRIO A — 5 Técnicas de Regressão =====

A avaliar modelo: LinearRegression...
Modelo: LinearRegression
 MAE:  4.147
 RMSE: 5.249
 R2:   0.930

A avaliar modelo: RandomForest...
Modelo: RandomForest
 MAE:  4.977
 RMSE: 6.319
 R2:   0.898

A avaliar modelo: DecisionTree...
Modelo: DecisionTree
 MAE:  7.361
 RMSE: 9.556
 R2:   0.769

A avaliar modelo: AdaBoost...
Modelo: AdaBoost
 MAE:  5.266
 RMSE: 6.539
 R2:   0.891

A avaliar modelo: GradientBoosting...
Modelo: GradientBoosting
 MAE:  4.682
 RMSE: 6.015
 R2:   0.907


Resultados finais Cenário A:
  Cenário            Modelo       MAE      RMSE        R2
0       A  LinearRegression  4.146984  5.248858  0.929666
1       A      RandomForest  4.977377  6.319204  0.898425
2       A      DecisionTree  7.361122  9.555721  0.768601
3       A          AdaBoost  5.266477  6.538517  0.891341
4       A  GradientBoosting  4.681670  6.014649  0.907344


In [6]:
df_results = pd.DataFrame(results)
print("\nResultados finais Cenário A:")
print(df_results)



Resultados finais Cenário A:
  Cenário            Modelo       MAE      RMSE        R2
0       A  LinearRegression  4.146984  5.248858  0.929666
1       A      RandomForest  4.977377  6.319204  0.898425
2       A      DecisionTree  7.361122  9.555721  0.768601
3       A          AdaBoost  5.266477  6.538517  0.891341
4       A  GradientBoosting  4.681670  6.014649  0.907344
